In [ ]:
%%capture
import os
from django_pandas.io import read_frame
from pathlib import Path
import pandas as pd

from dj_notebook import activate

env_file = os.environ["INTECOMM_ENV"]
documents_folder = os.environ["INTECOMM_DOCUMENTS_FOLDER"]
plus = activate(dotenv_file=env_file)

report_folder = Path(documents_folder)

In this notebook, explore conditions.

Subjects are categorized by the condition used to screen

Were there conditions not revealed during screening discovered at baseline?
Does anyone have a conditon that by itself makes them ineligible?

In [ ]:
df_main = pd.read_csv(Path("/Users/erikvw/Documents/ucl/protocols/intecomm/analysis/primary/") / "df_main_1858.csv")

df_main = df_main.drop(columns=['hiv_years_since_dx', 'hiv_timedelta_dx',
       'htn_years_since_dx', 'htn_timedelta_dx', 'dm_years_since_dx',
       'dm_timedelta_dx'])

In [ ]:
df1 = df_main.groupby(['hiv',"htn", "dm", "ncd"]).size().to_frame()
df1 = df1.reset_index()
df1.columns = ['hiv',"htn", "dm", "ncd", "count"]
df1

In [ ]:
df1[['hiv', "ncd", "count"]]

In [ ]:
df1[~(df1.hiv==1) & ((df1.htn==1) | (df1.dm==1))]["count"].sum()

In [ ]:
df1[~((df1.hiv==1) & ((df1.htn==1) | (df1.dm==1)))]["count"].sum()

In [ ]:
df_main[~((df_main.hiv==1) & ((df_main.htn==1) | (df_main.dm==1)))].subject_identifier

In [ ]:
df_main[~((df_main.hiv==1) & ((df_main.htn==1) | (df_main.dm==1)))].groupby("site_id").size().to_frame()

In [ ]:
from edc_pdutils.dataframes import get_crf
subject_identifiers = list(df_main.subject_identifier.unique())
opts = dict(
    subject_visit_model ="intecomm_subject.subjectvisit",
    subject_identifiers=subject_identifiers,
)
df_hiv_initial = get_crf(model ="intecomm_subject.hivinitialreview", **opts)
df_htn_initial = get_crf(model ="intecomm_subject.htninitialreview", **opts)
df_dm_initial = get_crf(model ="intecomm_subject.dminitialreview", **opts)


In [ ]:
df_dm_initial[df_dm_initial.dx_date.isna() & (df_dm_initial.dx_ago.notna())]

In [ ]:
# recalculate dx_date if from dx_ago
from edc_model import duration_to_date

def get_dx_date(s):
    if pd.isna(s["dx_date"]) and not pd.isna(s["dx_ago"]):
        dx_calculated_date = duration_to_date(s["dx_ago"], s["visit_datetime"])
        return dx_calculated_date
    return s["dx_date"]

df_hiv_initial["dx_date"] = df_hiv_initial.apply(get_dx_date, axis=1)
df_htn_initial["dx_date"] = df_htn_initial.apply(get_dx_date, axis=1)
df_dm_initial["dx_date"] = df_dm_initial.apply(get_dx_date, axis=1)


In [ ]:
df_hiv_initial["hiv_timedelta_dx"] =  (pd.to_datetime(df_hiv_initial["visit_datetime"]) - pd.to_datetime(df_hiv_initial["dx_date"]))
df_htn_initial["htn_timedelta_dx"] =  (pd.to_datetime(df_htn_initial["visit_datetime"]) - pd.to_datetime(df_htn_initial["dx_date"]))
df_dm_initial["dm_timedelta_dx"] =  (pd.to_datetime(df_dm_initial["visit_datetime"]) - pd.to_datetime(df_dm_initial["dx_date"]))


In [ ]:
df_hiv_initial["hiv_years_since_dx"] = df_hiv_initial["hiv_timedelta_dx"].dt.days / 365
df_htn_initial["htn_years_since_dx"] = df_htn_initial["htn_timedelta_dx"].dt.days / 365
df_dm_initial["dm_years_since_dx"] = df_dm_initial["dm_timedelta_dx"].dt.days / 365

df_delta = pd.merge(df_hiv_initial[["subject_identifier","hiv_years_since_dx", "hiv_timedelta_dx"]], df_htn_initial[["subject_identifier","htn_years_since_dx", "htn_timedelta_dx"]], on="subject_identifier", how="outer")
df_delta = df_delta.merge(df_dm_initial[["subject_identifier","dm_years_since_dx", "dm_timedelta_dx"]], on="subject_identifier", how="outer")


In [ ]:
df_main = df_main.merge(df_delta, on="subject_identifier", how="left")

In [ ]:
df_main.columns

In [ ]:
# maybe should be removed, HIV only?
df_main[(df_main.hiv_only==1) & (df_main.hiv_timedelta_dx<pd.Timedelta(days=182))]

In [ ]:
# maybe should be removed, NCD?
df_main[(df_main.ncd==1) & (df_main.htn_timedelta_dx<pd.Timedelta(days=182)) & (df_main.dm_timedelta_dx<pd.Timedelta(days=182))]

In [ ]:
# has no dx_date
# are any dm (ncd) without DX DATE
# if so, they must have been reported at screening but never confirmed
# this has been confirmed, there is no record after baseline of a
# diabetes diagnosis
df_main.loc[(df_main.dm==1) & (df_main.dm_years_since_dx.isna()), "dm"]  = 0
df_main.dm.value_counts()


In [ ]:
# has no dx_date
# are any htn (ncd) without DX DATE
# if so, they must have been reported at screening but never confirmed
# these have been confirmed, there is no record after baseline of any
# hypertension diagnosis

# df_main[(df_main.htn==1) & (df_main.htn_years_since_dx.isna())]
df_main.loc[(df_main.htn==1) & (df_main.htn_years_since_dx.isna()), "htn"]  = 0
df_main.htn.value_counts()


In [ ]:
# every HIV has a DX DATE
df_main[(df_main.hiv==1) & (df_main.hiv_years_since_dx.isna())]


In [ ]:
# df_main.hiv.value_counts()
df_hiv = df_main.hiv.value_counts().to_frame().reset_index()
df_hiv["cond"] = "hiv"
df_hiv = df_hiv[df_hiv.hiv==1][["cond", "count"]]

In [ ]:
df_main.htn.value_counts()
df_htn = df_main.htn.value_counts().to_frame().reset_index()
df_htn["cond"] = "htn"
df_htn = df_htn[df_htn.htn==1][["cond", "count"]]

In [ ]:
df_dm = df_main.dm.value_counts().to_frame().reset_index()
df_dm["cond"] = "dm"
df_dm = df_dm[df_dm.dm==1][["cond", "count"]]

In [ ]:
# thses are total diagnoses, where some people have multiple diagnoses
df_cond = pd.concat([df_hiv, df_htn, df_dm])
df_cond["total"] = df_cond["count"].sum()
df_cond["prop"] = df_cond["count"]/df_cond["total"]
df_cond

In [ ]:
df_main.groupby(["hiv", "ncd"]).size().to_frame().reset_index()

In [ ]:
# df_main.to_csv(Path("/Users/erikvw/Documents/ucl/protocols/intecomm/analysis/primary/") / "df_main_1858.csv", index=False)